In [1]:
import copy
from typing import Callable

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


from recommender_dl import build_mlp, move_tensors, per_var_temporal_split

In [2]:
class UserTower(nn.Module):
    def __init__(
        self,
        users_num: int,
        ages_num: int,
        genders_num: int,
        jobs_num: int,
        user_dim=16,
        age_dim=3,
        gender_dim=2,
        job_dim=4,
        hidden_dims=[64, 32],
        output_dim=16,
        dropout_ratio=0.2,
    ) -> None:
        super().__init__()
        
        self.user_embedding = nn.Embedding(
            users_num + 1, 
            user_dim,
        )
        self.age_embedding = nn.Embedding(
            ages_num + 1, 
            age_dim,
        )
        self.gender_embedding = nn.Embedding(
            genders_num + 1, 
            gender_dim,
        )
        self.job_embedding = nn.Embedding(
            jobs_num + 1, 
            job_dim,
        )
        
        input_dim = user_dim + age_dim + gender_dim + job_dim
        
        self.mlp = build_mlp(
            input_dim, 
            hidden_dims,
            output_dim,
            dropout_ratio,
        )

    def forward(
        self, 
        user_ids: torch.Tensor, 
        ages: torch.Tensor, 
        genders: torch.Tensor, 
        jobs: torch.Tensor,
    ) -> torch.Tensor:
        user_vector = self.user_embedding(user_ids)
        age_vector = self.age_embedding(ages)
        gender_vector = self.gender_embedding(genders)
        job_vector = self.job_embedding(jobs)
        
        cat_features = torch.cat(
            [
                user_vector, 
                age_vector, 
                gender_vector, 
                job_vector,
            ], 
            dim=1,
        )
        return self.mlp(cat_features)


class MovieTower(nn.Module):
    def __init__(
        self,
        movies_num: int,
        genres_num: int,
        movie_dim=16,
        genre_dim=4,
        hidden_dims=[64, 32],
        output_dim=16,
        dropout_ratio=0.2,
    ) -> None:
        super().__init__()
        
        self.movie_embedding = nn.Embedding(
            movies_num + 1, 
            movie_dim,
        )
        self.genre_projector = nn.Linear(
            genres_num, 
            genre_dim, 
            bias=False,
        )
        
        input_dim = movie_dim + genre_dim
        
        self.mlp = build_mlp(
            input_dim, 
            hidden_dims,
            output_dim,
            dropout_ratio,
        )

    def forward(
        self, 
        movie_ids: torch.Tensor, 
        genres: torch.Tensor,
    ) -> torch.Tensor:
        movie_vector = self.movie_embedding(movie_ids)
        genre_vector = self.genre_projector(genres)
        
        cat_features = torch.cat(
            [
                movie_vector, 
                genre_vector,
            ], 
            dim=1,
        )
        return self.mlp(cat_features)


class TwoTowerModel(nn.Module):
    def __init__(
        self,
        user_tower: UserTower,
        movie_tower: MovieTower,
        normalize_outputs=True,
    ) -> None:
        super().__init__()
        self.user_tower = user_tower
        self.movie_tower = movie_tower
        self.normalize_outputs = normalize_outputs

    def forward(
        self,
        user_ids: torch.Tensor,
        movie_ids: torch.Tensor,
        ages: torch.Tensor,
        genders: torch.Tensor,
        jobs: torch.Tensor,
        genres: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        
        user_vectors = self.user_tower(
            user_ids, 
            ages, 
            genders, 
            jobs,
        )
        movie_vectors = self.movie_tower(
            movie_ids, 
            genres,
        )
        
        if self.normalize_outputs:
            user_vectors = F.normalize(
                user_vectors, 
                p=2, 
                dim=1,
            )
            movie_vectors = F.normalize(
                movie_vectors, 
                p=2, 
                dim=1,
            )
        
        return user_vectors, movie_vectors

In [3]:
def get_all_movie_embeddings(
    two_towers: TwoTowerModel,
    all_movie_ids: torch.Tensor,
    all_genres: torch.Tensor,
    device: torch.device,
) -> torch.Tensor:
    
    movie_vectors = two_towers.movie_tower(
        all_movie_ids.to(device),
        all_genres.to(device),
    )
    
    if two_towers.normalize_outputs:
        movie_vectors = F.normalize(
            movie_vectors, 
            p=2, 
            dim=1,
        )
        
    return movie_vectors


def get_batch_user_embeddings(
    two_towers: TwoTowerModel,
    tensors: dict[str, torch.Tensor],
) -> torch.Tensor:
    
    user_vectors = two_towers.user_tower(
        tensors["user_ids"],
        tensors["ages"],
        tensors["genders"],
        tensors["jobs"],
    )

    if two_towers.normalize_outputs:
        user_vectors = F.normalize(
            user_vectors, 
            p=2, 
            dim=1,
        )
        
    return user_vectors


def calculate_batch_hits(
    user_vectors: torch.Tensor,
    all_movie_vectors: torch.Tensor,
    all_movie_ids: torch.Tensor,
    target_movie_ids: torch.Tensor,
    k: int,
    device: torch.device,
) -> tuple[float, int]:
    
    similarity_scores = torch.matmul(
        user_vectors, 
        all_movie_vectors.T,
    )
    
    _, top_k = torch.topk(
        similarity_scores, 
        k=k, 
        dim=1,
    )
    
    target_movie_ids = target_movie_ids.view(-1, 1)
    
    retrieved_movie_ids = all_movie_ids[top_k].to(
        device,
    )
    
    match = (retrieved_movie_ids == target_movie_ids)
    
    return match.sum().item(), target_movie_ids.size(0)


def recall_at_k(
    two_towers: TwoTowerModel,
    dataloader: DataLoader,
    var_key_map: dict[str, str],
    all_movie_ids: torch.Tensor,
    all_genres: torch.Tensor,
    device: torch.device,
    k=100,
) -> float:
    
    two_towers.eval()
    hits = 0.0
    total = 0.0
    
    with torch.no_grad():
        all_movie_vectors = get_all_movie_embeddings(
            two_towers=two_towers,
            all_movie_ids=all_movie_ids,
            all_genres=all_genres,
            device=device,
        )
        
        for batch in dataloader:
            tensors = move_tensors(
                var_key_map, 
                batch, 
                device,
            )
            
            user_vectors = get_batch_user_embeddings(
                two_towers=two_towers,
                tensors=tensors,
            )
            
            batch_hits, batch_total = calculate_batch_hits(
                user_vectors=user_vectors,
                all_movie_vectors=all_movie_vectors,
                all_movie_ids=all_movie_ids,
                target_movie_ids=tensors["movie_ids"],
                k=k,
                device=device,
            )
            
            hits += batch_hits
            total += batch_total
    
    if total > 0:
        return hits / total
    return 0

In [4]:
def build_interaction_matrix(
    user_indices: torch.Tensor,
    movie_indices: torch.Tensor,
    max_user_id: int,
    max_movie_id: int,
) -> torch.Tensor:
    
    interaction_matrix = torch.zeros(
        max_user_id + 1, 
        max_movie_id + 1, 
        dtype=torch.bool,
    )
    
    interaction_matrix[user_indices, movie_indices] = True
    
    return interaction_matrix


def info_nce(
    user_vectors: torch.Tensor,
    movie_vectors: torch.Tensor,
    user_ids: torch.Tensor,
    movie_ids: torch.Tensor,
    interaction_matrix: torch.Tensor | None = None,
    check_interactions=True,
    temperature=1.0,
) -> torch.Tensor:
    
    logits = torch.matmul(user_vectors, movie_vectors.T) / temperature
    
    device = logits.device
    
    eye_mask = torch.eye(
        logits.size(0), 
        dtype=torch.bool, 
        device=device,
    )
    
    if check_interactions and interaction_matrix is not None:
        batch_mask = interaction_matrix[user_ids][:, movie_ids].to(
            device,
        )
        collision_mask = batch_mask & ~eye_mask
    else:
        movie_mask = (
            movie_ids.unsqueeze(1) == movie_ids.unsqueeze(0)
        )
        user_mask = (
            user_ids.unsqueeze(1) == user_ids.unsqueeze(0)
        )
        collision_mask = (movie_mask | user_mask) & ~eye_mask
        
    logits.masked_fill_(
        collision_mask, 
        -1e9,
    )
    
    targets = torch.arange(
        logits.size(0), 
        device=device,
    )
    
    return F.cross_entropy(
        logits, 
        targets,
    )


def _run_retrieval_epoch(
    model: TwoTowerModel,
    dataloader: DataLoader[dict[str, torch.Tensor]],
    var_key_map: dict[str, str],
    device: torch.device,
    is_train: bool,
    embedding_loss: Callable,
    optimizer: optim.Optimizer | None = None,
    **loss_kwargs,
) -> float:
    if is_train:
        model.train()
    else:
        model.eval()
    
    epoch_loss = 0.0
    dataset_len = 0
    
    with torch.set_grad_enabled(is_train):
        for batch in dataloader:
            tensors = move_tensors(var_key_map, batch, device)

            user_vectors, movie_vectors = model(**tensors)
            
            loss = embedding_loss(
                user_vectors,
                movie_vectors,
                tensors["user_ids"],
                tensors["movie_ids"],
                **loss_kwargs,
            )
            
            if is_train and optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
            batch_size = user_vectors.size(0)
            epoch_loss += loss.item() * batch_size
            dataset_len += batch_size
            
    return epoch_loss / dataset_len


def train_retriever(
    model: TwoTowerModel,
    train_loader: DataLoader,
    val_loader: DataLoader,
    all_movie_ids: torch.Tensor,
    all_genres: torch.Tensor,
    var_key_map: dict[str, str],
    embedding_loss: Callable,
    epochs=10_000,
    learning_rate=0.001,
    patience=10,
    tol=1e-4,
    weight_decay=1e-4,
    device_name="cpu",
    recall_k=100,
    **loss_kwargs,
) -> None:
    
    device = torch.device(device_name)
    model = model.to(device)
    
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    
    best_val_recall = -1
    to_beat_recall = -1
    no_improve_count = 0
    best_weights = None
    
    for epoch in range(epochs):
        avg_train_loss = _run_retrieval_epoch(
            model=model,
            dataloader=train_loader,
            var_key_map=var_key_map,
            device=device,
            is_train=True,
            embedding_loss=embedding_loss,
            optimizer=optimizer,
            **loss_kwargs,
        )
        
        avg_val_loss = _run_retrieval_epoch(
            model=model,
            dataloader=val_loader,
            var_key_map=var_key_map,
            device=device,
            is_train=False,
            embedding_loss=embedding_loss,
            **loss_kwargs,
        )

        val_recall = recall_at_k(
            two_towers=model,
            dataloader=val_loader,
            var_key_map=var_key_map,
            all_movie_ids=all_movie_ids,
            all_genres=all_genres,
            device=device,
            k=recall_k,
        )
        
        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Train Loss: {avg_train_loss:.4f}, "
            f"Val Loss: {avg_val_loss:.4f}. ",
            f"Val recall@{recall_k}: {val_recall:.4f}"
        )
        
        if val_recall > best_val_recall:
            best_val_recall = val_recall
            best_weights = copy.deepcopy(
                model.state_dict(),
            )

        if val_recall > (to_beat_recall - tol):
            to_beat_recall = val_recall
            no_improve_count = 0
        else:
            no_improve_count += 1
            
        if no_improve_count >= patience:
            break
    
    if best_weights is not None:
        model.load_state_dict(best_weights)
        print(
            "Best weights restored. "
            f"Validation recall@{recall_k} is {best_val_recall:.4f}",
        )


In [5]:
with duckdb.connect("/home/maria-khanina/mkgit/movies_project/data/database.duckdb", read_only=True) as db_conn:
    ratings_df = db_conn.execute("""
        SELECT *
        FROM ratings
            INNER JOIN movies USING (movie_id)
            INNER JOIN users USING (user_id)
    """).df()

In [6]:
genres_encoded = ratings_df["genres"].str.get_dummies(sep="|")
genres_cols = genres_encoded.columns.to_list()
genres_num = len(genres_cols)
ratings_df = pd.concat([ratings_df, genres_encoded], axis=1)

In [7]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)

cat_cols = ["gender"]

ratings_df[cat_cols] = encoder.fit_transform(ratings_df[cat_cols]) + 1
ratings_df["occupation"] = ratings_df["occupation"] + 1

In [8]:
max_user = int(ratings_df["user_id"].max())
max_movie = int(ratings_df["movie_id"].max())
max_age = int(ratings_df["age"].max())
max_gender = int(ratings_df["gender"].max())
max_job = int(ratings_df["occupation"].max())

In [9]:
train_all_df, test_df = per_var_temporal_split(
    X=ratings_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

train_big_df, outer_val_df = per_var_temporal_split(
    train_all_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

train_df, val_df = per_var_temporal_split(
    train_big_df,
    var="user_id",
    time_col="timestamp",
    max_test_ratio=0.1,
    test_max_per_var=8,
    test_min_per_var=3,
)

In [10]:
var_key_map = {
    "user_ids": "user_id",
    "movie_ids": "movie_id",
    "ages": "age",
    "genders": "gender",
    "jobs": "job",
    "genres": "genre",
}

In [11]:
class PositiveRatings(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        genres_cols: list[str],
        min_rating: float = 4.0,
    ) -> None:
        super().__init__()
        
        positive_df = df[df["rating"] >= min_rating].reset_index(
            drop=True,
        )
        
        self.user_ids = torch.tensor(
            positive_df["user_id"].values,
            dtype=torch.long,
        )
        self.movie_ids = torch.tensor(
            positive_df["movie_id"].values, 
            dtype=torch.long,
        )
        self.ages = torch.tensor(
            positive_df["age"].values, 
            dtype=torch.long,
        )
        self.genders = torch.tensor(
            positive_df["gender"].values, 
            dtype=torch.long,
        )
        self.jobs = torch.tensor(
            positive_df["occupation"].values, 
            dtype=torch.long,
        )
        self.genres = torch.tensor(
            positive_df[genres_cols].values, 
            dtype=torch.float32,
        )

    def __len__(self) -> int:
        return len(self.user_ids)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "user_id": self.user_ids[idx],
            "movie_id": self.movie_ids[idx],
            "age": self.ages[idx],
            "gender": self.genders[idx],
            "job": self.jobs[idx],
            "genre": self.genres[idx],
        }

In [12]:
train_dataset = PositiveRatings(train_df, genres_cols)
val_dataset = PositiveRatings(val_df, genres_cols)
outer_val_dataset = PositiveRatings(outer_val_df, genres_cols)

train_loader = DataLoader(
    train_dataset, 
    batch_size=512,
    shuffle=True,
    num_workers=4,
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=512,
    shuffle=False,
    num_workers=4,
)

outer_val_loader = DataLoader(
    outer_val_dataset, 
    batch_size=512,
    shuffle=False,
    num_workers=4,
)

In [13]:
interactions = build_interaction_matrix(
    user_indices=train_dataset.user_ids,
    movie_indices=train_dataset.movie_ids,
    max_user_id=max_user,
    max_movie_id=max_movie,
)

In [14]:
user_tower = UserTower(
    users_num=max_user,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
)
    
movie_tower = MovieTower(
    movies_num=max_movie,
    genres_num=genres_num,
)

retriever = TwoTowerModel(
    user_tower=user_tower,
    movie_tower=movie_tower,
    normalize_outputs=True,
)

In [15]:
def get_unique_movies(
    df: pd.DataFrame,
    genres_cols: list[str],
) -> tuple[torch.Tensor, torch.Tensor]:
    
    unique_movies_df = df.drop_duplicates(
        subset=["movie_id"],
    ).sort_values(
        by="movie_id",
    ).reset_index(
        drop=True,
    )
    
    movie_ids = torch.tensor(
        unique_movies_df["movie_id"].values,
        dtype=torch.long,
    )
    
    movie_genres = torch.tensor(
        unique_movies_df[genres_cols].values,
        dtype=torch.float32,
    )
    
    return movie_ids, movie_genres

unique_movie_ids, unique_movie_genres = get_unique_movies(
    train_df,
    genres_cols,
)

In [16]:
train_retriever(
    model=retriever,
    train_loader=train_loader,
    val_loader=val_loader,
    all_movie_ids=unique_movie_ids,
    all_genres=unique_movie_genres,
    var_key_map=var_key_map,
    embedding_loss=info_nce,
    epochs=10_000,
    learning_rate=0.001,
    patience=10,
    tol=0.0001,
    weight_decay=0.0001,
    device_name="cpu",
    recall_k=100,
    interaction_matrix=interactions,
    check_interactions=True,
    temperature=0.1,
)

Epoch 1/10000, Train Loss: 6.0226, Val Loss: 6.0939.  Val recall@100: 0.0644
Epoch 2/10000, Train Loss: 5.9376, Val Loss: 6.0666.  Val recall@100: 0.0863
Epoch 3/10000, Train Loss: 5.8617, Val Loss: 6.0242.  Val recall@100: 0.1086
Epoch 4/10000, Train Loss: 5.7792, Val Loss: 5.9804.  Val recall@100: 0.1277
Epoch 5/10000, Train Loss: 5.7185, Val Loss: 5.9481.  Val recall@100: 0.1380
Epoch 6/10000, Train Loss: 5.6721, Val Loss: 5.9175.  Val recall@100: 0.1481
Epoch 7/10000, Train Loss: 5.6351, Val Loss: 5.8905.  Val recall@100: 0.1518
Epoch 8/10000, Train Loss: 5.6052, Val Loss: 5.8716.  Val recall@100: 0.1535
Epoch 9/10000, Train Loss: 5.5793, Val Loss: 5.8507.  Val recall@100: 0.1600
Epoch 10/10000, Train Loss: 5.5561, Val Loss: 5.8378.  Val recall@100: 0.1625
Epoch 11/10000, Train Loss: 5.5344, Val Loss: 5.8174.  Val recall@100: 0.1633
Epoch 12/10000, Train Loss: 5.5143, Val Loss: 5.8044.  Val recall@100: 0.1705
Epoch 13/10000, Train Loss: 5.4948, Val Loss: 5.7929.  Val recall@100: 0.

In [25]:
for k in [10, 50, 100, 200, 500]:
    val_recall = recall_at_k(
        retriever,
        outer_val_loader,
        var_key_map,
        unique_movie_ids,
        unique_movie_genres,
        torch.device("cpu"),
        k=k,
    )
    print(f"Outer validation recall@{k}: {val_recall:.4f}")


Outer validation recall@10: 0.0199
Outer validation recall@50: 0.0928
Outer validation recall@100: 0.1635
Outer validation recall@200: 0.2695
Outer validation recall@500: 0.4759


In [28]:
big_user_tower = UserTower(
    users_num=max_user,
    ages_num=max_age,
    genders_num=max_gender,
    jobs_num=max_job,
    user_dim=32,
    age_dim=4,
    job_dim=8,
    output_dim=32,
    hidden_dims=[128, 64],
    dropout_ratio=0.3,
)
    
big_movie_tower = MovieTower(
    movies_num=max_movie,
    genres_num=genres_num,
    movie_dim=32,
    genre_dim=8,
    output_dim=32,
    hidden_dims=[128, 64],
    dropout_ratio=0.3,
)

big_retriever = TwoTowerModel(
    user_tower=big_user_tower,
    movie_tower=big_movie_tower,
    normalize_outputs=True,
)

In [29]:
train_retriever(
    model=big_retriever,
    train_loader=train_loader,
    val_loader=val_loader,
    all_movie_ids=unique_movie_ids,
    all_genres=unique_movie_genres,
    var_key_map=var_key_map,
    embedding_loss=info_nce,
    epochs=10_000,
    learning_rate=0.001,
    patience=10,
    tol=0.0001,
    weight_decay=0.0001,
    device_name="cpu",
    recall_k=100,
    interaction_matrix=interactions,
    check_interactions=True,
    temperature=0.05,
)

Epoch 1/10000, Train Loss: 6.0684, Val Loss: 6.0986.  Val recall@100: 0.0672
Epoch 2/10000, Train Loss: 5.9318, Val Loss: 6.0625.  Val recall@100: 0.0991
Epoch 3/10000, Train Loss: 5.8323, Val Loss: 5.9944.  Val recall@100: 0.1243
Epoch 4/10000, Train Loss: 5.7287, Val Loss: 5.9503.  Val recall@100: 0.1433
Epoch 5/10000, Train Loss: 5.6507, Val Loss: 5.8952.  Val recall@100: 0.1513
Epoch 6/10000, Train Loss: 5.5877, Val Loss: 5.8569.  Val recall@100: 0.1628
Epoch 7/10000, Train Loss: 5.5364, Val Loss: 5.8223.  Val recall@100: 0.1733
Epoch 8/10000, Train Loss: 5.4958, Val Loss: 5.8005.  Val recall@100: 0.1811
Epoch 9/10000, Train Loss: 5.4617, Val Loss: 5.7785.  Val recall@100: 0.1826
Epoch 10/10000, Train Loss: 5.4329, Val Loss: 5.7633.  Val recall@100: 0.1890
Epoch 11/10000, Train Loss: 5.4079, Val Loss: 5.7469.  Val recall@100: 0.1907
Epoch 12/10000, Train Loss: 5.3853, Val Loss: 5.7367.  Val recall@100: 0.1933
Epoch 13/10000, Train Loss: 5.3649, Val Loss: 5.7253.  Val recall@100: 0.

In [30]:
for k in [10, 50, 100, 200, 500]:
    val_recall = recall_at_k(
        big_retriever,
        outer_val_loader,
        var_key_map,
        unique_movie_ids,
        unique_movie_genres,
        torch.device("cpu"),
        k=k,
    )
    print(f"Outer validation recall@{k}: {val_recall:.4f}")

Outer validation recall@10: 0.0196
Outer validation recall@50: 0.0987
Outer validation recall@100: 0.1739
Outer validation recall@200: 0.2827
Outer validation recall@500: 0.4906
